In [ ]:
"""100%-stacked-bar version of detection coutcome columns from Data_2_Sample_Metadata.xlsx, 
per virus, but each bar normalized to its own total (100%) instead of raw sample-ID counts. 
"""

#Import required packages
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

#Input data and output file name
DATA_FILE = "Data_2_Sample_Metadata_Final.xlsx"
OUT_PATH = "fig2_samples_tested_by_virus_.png"

VIRUS_COL = "Respiratory Virus"
TESTED_COL = "Samples tested by both qPCR and hybrid-capture sequencing"

CATEGORIES = ["Detected by both methods", "Detected by qPCR only", "Detected by sequencing only", "Detected by neither"]

LABELS = {
    "Detected by both methods": "Detected in both qPCR and Sequencing",
    "Detected by qPCR only": "qPCR only",
    "Detected by sequencing only": "Sequencing only",
    "Detected by neither": "Detected in neither qPCR nor Sequencing",
}

COLORS = {
    "Detected by both methods": "#8ac926", #Green
    "Detected by qPCR only": "#f4f1bb", #Cream
    "Detected by sequencing only": "#ed6a5a", #Orange
    "Detected by neither": "#dce0d9", #Gray
}

BAR_WIDTH = 0.42
MIN_INSIDE_PCT = 8.0  # segments at or above this share get their label centered inside the bar
MIN_GAP = 6.5  # minimum vertical spacing between stacked external labels, in percentage points
LEADER_START, LEADER_END, TEXT_X = 0.04, 0.16, 0.20  # x-offsets from the bar's right edge (data units = virus slots)


def main():
    
    df = pd.read_excel(DATA_FILE, sheet_name="Sheet1")
    df.columns = [c.strip() for c in df.columns]

    x = np.arange(len(df))
    totals = df[TESTED_COL].to_numpy()

    fig, ax = plt.subplots(figsize=(13, 8))

    bottom = np.zeros(len(df))
    bar_segments = [[] for _ in range(len(df))]
    for cat in CATEGORIES:
        counts = df[cat].to_numpy()
        pcts = counts / totals * 100.0
        bars = ax.bar(x, pcts, bottom=bottom, width=BAR_WIDTH, color=COLORS[cat],
                       edgecolor="white", linewidth=1.5, label=LABELS[cat], zorder=3)
        for i, (bar, pct, n, b) in enumerate(zip(bars, pcts, counts, bottom)):
            if pct <= 0:
                continue
            center = b + pct / 2
            if pct >= MIN_INSIDE_PCT:
                ax.text(bar.get_x() + bar.get_width() / 2, center, f"{pct:.0f}%\n(n={n:,})",
                        ha="center", va="center", fontsize=10, fontweight="bold", color="black", zorder=4)
            else:
                bar_segments[i].append({"cat": cat, "pct": pct, "n": n, "center": center})
        bottom += pcts
    
    # Total N tested, above each bar.
    for xi, total in zip(x, totals):
        ax.text(xi, 101.5, f"N={total:,}", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color="#333333", zorder=5)
    
    # External leader-line labels for the too-thin-to-fit segments, de-collided bottom-to-top per bar.
    for i, segs in enumerate(bar_segments):
        bar_right = x[i] + BAR_WIDTH / 2
        label_ys = [s["center"] for s in segs]
        MIN_FIRST_Y = 3.0
        if label_ys and label_ys[0] < MIN_FIRST_Y:
            label_ys[0] = MIN_FIRST_Y
        for j in range(1, len(label_ys)):
            if label_ys[j] - label_ys[j - 1] < MIN_GAP:
                label_ys[j] = label_ys[j - 1] + MIN_GAP

        for seg, label_y in zip(segs, label_ys):
            ax.plot(
                [bar_right + LEADER_START, bar_right + LEADER_END],
                [seg["center"], label_y],
                color=COLORS[seg["cat"]], linewidth=1.3, zorder=4,
                clip_on=False, solid_capstyle="round",
            )
            ax.text(
                bar_right + TEXT_X, label_y, f"{seg['pct']:.0f}%\n(n={seg['n']:,})",
                ha="left", va="center", fontsize=10,
                color="black", clip_on=False, zorder=5,fontweight="bold"
            )

    ax.set_xticks(x)
    ax.set_xticklabels(df[VIRUS_COL], fontsize=13)
    ax.set_xlabel("Respiratory virus", fontsize=16, fontweight="bold")
    ax.set_ylabel("% of samples", fontsize=16)
    ax.set_xlim(-0.6, len(df) - 1 + BAR_WIDTH / 2 + 0.95)
    ax.set_ylim(0, 108)
    ax.set_yticks(range(0, 101, 10))
    ax.set_yticklabels([f"{v}%" for v in range(0, 101, 10)])
    ax.grid(axis="y", color="#e0e0e0", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0)

    ax.set_title(
        "% of qPCR vs. Sequencing Detection Outcomes per Respiratory Virus\n"
        , fontsize=17, fontweight="bold", loc="center", pad=20,
    )

    ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2,
        frameon=False, fontsize=14,
    )

    fig.tight_layout(rect=[0, 0.06, 1, 1])
    fig.savefig(OUT_PATH, dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(OUT_PATH.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3) #Save as svg for future edit and pnaels in Inkscape
    plt.close(fig)
    print(f"wrote {OUT_PATH}")


if __name__ == "__main__":
    main()



wrote fig2b_samples_tested_by_virus_nodallas.png
